# Healthcare Analytics Dashboard — EDA & KPI Notebook
**Author:** Lohith Movva  
**Stack:** Python, PostgreSQL, Pandas, Matplotlib, Seaborn  
**Description:** Exploratory analysis of EHR, claims, and encounter data to validate ETL output and surface insights for dashboard development.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded.')

## 1. Load Data

In [ ]:
# Load from CSV (run etl/etl_pipeline.py first to generate these)
patients   = pd.read_csv('../data/dim_patients.csv')
claims     = pd.read_csv('../data/fact_claims.csv', parse_dates=['claim_date'])
encounters = pd.read_csv('../data/fact_encounters.csv', parse_dates=['admit_date'])
summary    = pd.read_csv('../data/kpi_monthly_summary.csv')

print(f'Patients:   {len(patients):,}')
print(f'Claims:     {len(claims):,}')
print(f'Encounters: {len(encounters):,}')
print(f'Monthly summary rows: {len(summary)}')

## 2. Data Quality Checks

In [ ]:
print('=== Null check ===')
for name, df in [('Patients', patients), ('Claims', claims), ('Encounters', encounters)]:
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'{name}:\n{nulls[nulls > 0]}\n')
    else:
        print(f'{name}: No nulls.')

print('\n=== Duplicate check ===')
print(f'Duplicate patients:   {patients.duplicated(subset="patient_id").sum()}')
print(f'Duplicate claims:     {claims.duplicated(subset="claim_id").sum()}')
print(f'Duplicate encounters: {encounters.duplicated(subset="encounter_id").sum()}')

## 3. Key Performance Indicators

In [ ]:
kpis = {
    'Avg Length of Stay (days)':   round(encounters['length_of_stay'].mean(), 2),
    '30-Day Readmission Rate (%)': round(encounters['readmitted_30d'].mean() * 100, 2),
    'Avg Cost per Encounter ($)':  round(encounters['cost_per_encounter'].mean(), 2),
    'Claims Denial Rate (%)':      round((~claims['approved']).mean() * 100, 2),
    'Total Billed ($M)':           round(claims['claim_amount'].sum() / 1e6, 2),
    'Total Approved ($M)':         round(claims['approved_amount'].sum() / 1e6, 2),
    'Chronic Patient %':           round(patients['is_chronic'].mean() * 100, 2),
}

print('\n======= DASHBOARD KPIs =======')
for k, v in kpis.items():
    print(f'  {k:<35} {v}')

## 4. Monthly Encounter Trend

In [ ]:
monthly = encounters.groupby('month').agg(
    encounters=('encounter_id','count'),
    avg_los=('length_of_stay','mean'),
    readmit_rate=('readmitted_30d','mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(monthly['month'], monthly['encounters'], color='#1A2E4A')
axes[0].set_title('Monthly Encounter Volume', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Encounters')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['month'], monthly['readmit_rate'] * 100, marker='o', color='#C0392B', linewidth=2)
axes[1].axhline(15, linestyle='--', color='gray', alpha=0.6, label='CMS Benchmark (15%)')
axes[1].set_title('30-Day Readmission Rate (%)', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Rate (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/monthly_trends.png', bbox_inches='tight')
plt.show()

## 5. Cost & LOS by Department

In [ ]:
dept = encounters.groupby('department').agg(
    avg_los=('length_of_stay','mean'),
    avg_cost=('cost_per_encounter','mean'),
    readmit_rate=('readmitted_30d','mean')
).reset_index().sort_values('avg_cost', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=dept, x='avg_cost', y='department', palette='Blues_d', ax=axes[0])
axes[0].set_title('Avg Cost per Encounter by Dept', fontweight='bold')
axes[0].set_xlabel('Avg Cost ($)')
axes[0].set_ylabel('')

sns.barplot(data=dept, x='readmit_rate', y='department', palette='Reds_d', ax=axes[1])
axes[1].set_title('Readmission Rate by Department', fontweight='bold')
axes[1].set_xlabel('Rate')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../data/dept_analysis.png', bbox_inches='tight')
plt.show()

## 6. Claims Analysis by Payer

In [ ]:
merged = claims.merge(patients[['patient_id','payer_type']], on='patient_id', how='left')
payer_claims = merged.groupby('payer_type').agg(
    total_claims=('claim_id','count'),
    total_billed=('claim_amount','sum'),
    total_approved=('approved_amount','sum'),
    denial_rate=('approved', lambda x: (~x).mean() * 100)
).reset_index().sort_values('total_billed', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(payer_claims))
width = 0.35
ax.bar([i - width/2 for i in x], payer_claims['total_billed'] / 1e6, width, label='Billed ($M)', color='#1A2E4A')
ax.bar([i + width/2 for i in x], payer_claims['total_approved'] / 1e6, width, label='Approved ($M)', color='#1a6b38')
ax.set_xticks(x)
ax.set_xticklabels(payer_claims['payer_type'])
ax.set_title('Claims: Billed vs Approved by Payer Type', fontweight='bold')
ax.set_ylabel('Amount ($M)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/payer_analysis.png', bbox_inches='tight')
plt.show()

print('\nDenial Rate by Payer:')
print(payer_claims[['payer_type','denial_rate']].to_string(index=False))

## 7. Patient Engagement — Chronic vs Non-Chronic

In [ ]:
enc_pat = encounters.merge(patients[['patient_id','is_chronic','age_group','payer_type']], on='patient_id', how='left')

chronic_summary = enc_pat.groupby('is_chronic').agg(
    encounters=('encounter_id','count'),
    avg_los=('length_of_stay','mean'),
    readmit_rate=('readmitted_30d','mean'),
    avg_cost=('cost_per_encounter','mean')
).reset_index()
chronic_summary['is_chronic'] = chronic_summary['is_chronic'].map({True:'Chronic', False:'Non-Chronic'})

print('Chronic vs Non-Chronic Patient Engagement:')
print(chronic_summary.to_string(index=False))